# 02 · Interfaces, local model fit and ranking modes — PSMA / GCPII (Q04609)

**Milestone 2.** This notebook builds on the survey workbook from notebook 01 and adds:

1. **Biological dimer interfaces.** The selected biological assembly is built from its operators. For every PSMA–PSMA pair the notebook records contacts, H-bonds, salt bridges, hydrophobic and aromatic contacts, buried surface area, composition, B-factors, validation outliers, and nearby glycans and ligands.
2. **Interface comparison** across all structures:
   - Jaccard similarity of residue sets and of contact pairs, against the consensus "canonical" interface;
   - interface RMSD and the change in relative orientation;
   - hierarchical clustering;
   - flags for noncanonical interfaces.
3. **Crystal-lattice contacts.** For example, is the canonical dimer present in the crystal even when the deposited assembly is a monomer, as in 4P44?
4. **Local experimental support** from the wwPDB validation report. X-ray structures use RSCC and RSRZ; cryo-EM structures use Q-scores relative to the resolution. This covers the active site, metal ligands, ligand contacts, entrance lid, glutarate sensor, interface, glycosylation sites, ligands and ions.
5. **Ten transparent ranking modes**, each with its own filters, criteria, weights and explanations.

Pocket fingerprints, conserved waters and variant mapping follow in Milestone 3, in the second half of this notebook.

**Run order:** setup → unit tests → validation set with sanity checks (gate) → full assessment → plots.

## 0 · Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path
REPO = "https://github.com/victorolatunde23-dev/psma_structural_modelling.git"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("psma_structural_modelling").exists() and not Path("src/psma_pipeline").exists():
    subprocess.run(["git", "clone", "-q", REPO], check=True)
if Path("psma_structural_modelling").exists():
    os.chdir("psma_structural_modelling")
if Path("../src/psma_pipeline").exists():
    os.chdir("..")
if IN_COLAB:
    subprocess.run(["git", "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "freesasa"], check=False)  # optional; numpy fallback otherwise
sys.path.insert(0, str(Path("src").resolve()))
# Forget any previously imported version of the pipeline (e.g. after `git pull` in a running session),
# so the code on disk is the code that runs.
for _m in [m for m in list(sys.modules) if m == "psma_pipeline" or m.startswith("psma_pipeline.")]:
    del sys.modules[_m]
import psma_pipeline
print("project root:", Path.cwd(), "| psma_pipeline", psma_pipeline.__version__)

## 1 · Configuration and the survey results from notebook 01

The notebooks share one Google Drive folder, `MyDrive/psma_structural_modelling/`, because Colab deletes its local disk on every restart. This notebook reads notebook 01's survey workbook from there, and reuses the same download cache.

**If the survey workbook is not found**, for example because notebook 01 was run before Drive storage existed, this notebook runs the survey itself (`AUTO_RUN_SURVEY = True`). That uses the same code as notebook 01 and takes roughly 10–20 minutes the first time. You can also upload an existing workbook and set `SURVEY` to its path.

> **Decision — read the workbook rather than re-running the survey.** The two stages are decoupled, so this notebook can be re-run, or its settings changed, without repeating notebook 01. It also documents exactly which survey results the assessment used.

In [ ]:
import pandas as pd
from psma_pipeline import load_config, setup_logging
from psma_pipeline.colab import storage_overrides
from psma_pipeline.assessment import load_survey_tables, run_assessment, save_assessment
from psma_pipeline.sanity import run_assessment_checks, critical_failures, run_sanity_checks
from psma_pipeline.sasa import resolve_method

setup_logging()
USE_DRIVE = True            # share results with the other notebooks via Google Drive
AUTO_RUN_SURVEY = True      # run the survey here if notebook 01's workbook is not found
cfg = load_config(overrides=storage_overrides(USE_DRIVE) or None)

SURVEY = cfg.output_file("survey_workbook")      # or e.g. Path("/content/Q04609_structures.xlsx") after uploading
if not Path(SURVEY).exists():
    assert AUTO_RUN_SURVEY, f"{SURVEY} not found: run notebook 01 or set SURVEY to an uploaded workbook"
    print(f"{SURVEY} not found: running the survey now (same code as notebook 01) ...")
    from psma_pipeline.survey import run_survey, save_survey
    _surv = run_survey(cfg)
    save_survey(cfg, _surv, extra_sheets={"SanityChecks": run_sanity_checks(_surv.tables, cfg)})
tables = load_survey_tables(SURVEY)
print(f"survey: {SURVEY}, {len(tables['Summary'])} entries")
print("SASA method:", resolve_method(cfg.get("interfaces.sasa.method")))

## A · Unit tests

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(r.stdout[-2500:], r.stderr[-1500:])
assert r.returncode == 0, "Unit tests failed: fix before running the assessment."

## B · Validation set

The validation set is chosen from the survey results, so it works whether or not notebook 01's own validation set is available. It contains:
- notebook 01's validation set and the configured fixed IDs (3BHX, 2XEG, 2OOT, 9S7U);
- the **three top-ranked author-defined dimers** (these define the canonical interface);
- **every entry whose assembly is a monomer** (4P44), to test the crystal-lattice search;
- the best **cryo-EM** entry.

> **Gate.** The full run is stopped by a critical failure:
> - a dimer assembly with zero interface contacts;
> - an empty canonical interface.
>
> Other checks report problems without stopping the run:
> - every contacting interface buries surface;
> - a ligand without a fit metric is never called supported;
> - EM metrics appear only for cryo-EM entries.

In [ ]:
from psma_pipeline.sanity import assessment_validation_ids
selection = assessment_validation_ids(tables, cfg)
display(selection)
val_ids = selection["PDB ID"].tolist()
val = run_assessment(cfg, tables, ids=val_ids)
val_checks = run_assessment_checks(val.tables, tables["Summary"][tables["Summary"]["PDB ID"].isin(val_ids)])
display(val_checks)
display(val.tables["AssessmentSummary"].set_index("PDB ID").loc[val_ids, [
    "Monomer / dimer", "Dimer interface area (Å²)", "Interface residue count", "Canonical interface similarity",
    "Interface complete", "Canonical dimer in crystal lattice", "Active-site local support", "Ligand local support"]])
save_assessment(cfg, val, tag="validation")
assert critical_failures(val_checks).empty, "Critical assessment checks failed: full run stopped."

## C · Full assessment

This step takes about 2–5 seconds per entry. Most of that is solvent-accessible surface area, plus validation-report downloads on the first run.

In [ ]:
res = run_assessment(cfg, tables)
checks = run_assessment_checks(res.tables, tables["Summary"])
display(checks["Status"].value_counts().to_frame("checks"))
display(checks[checks["Status"] != "PASS"])
paths = save_assessment(cfg, res)
print("workbook:", paths["workbook"])
T = res.tables
display(res.issues.to_frame().head(30))

## Dimer interfaces

**Canonical interface:** residues present in at least half of the interfaces of entries with an *author-defined* dimer assembly (`interfaces.canonical.min_fraction`).

> **Decision — a consensus definition.** Defining the canonical interface by consensus, rather than from one reference structure, stops a single unusual structure from defining "normal". The reference used for interface RMSD is the top-ranked entry that has an interface.

In [ ]:
display(T["CanonicalInterface"])
s = T["AssessmentSummary"]
cols = ["PDB ID", "Method", "Resolution (Å)", "Monomer / dimer", "Dimer interface area (Å²)", "Interface residue count",
        "Canonical interface similarity", "Interface complete", "Noncanonical interface flag", "Interface cluster",
        "Canonical dimer in crystal lattice"]
display(s[[c for c in cols if c in s]].sort_values("Canonical interface similarity"))

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform

J, ids = res.jaccard_matrix, [i.split(":")[0] for i in res.matrix_ids]
if len(ids) > 1:
    D = 1 - np.nan_to_num(J); np.fill_diagonal(D, 0); D = (D + D.T) / 2
    Z = linkage(squareform(D, checks=False), method=cfg.get("interfaces.clustering.method"))
    fig = plt.figure(figsize=(14, 11))
    ax_d = fig.add_axes([0.05, 0.1, 0.15, 0.8]); dn = dendrogram(Z, orientation="left", no_labels=True, ax=ax_d,
                                                                 color_threshold=cfg.get("interfaces.clustering.distance_threshold"))
    ax_d.axis("off")
    order = dn["leaves"][::-1]
    ax = fig.add_axes([0.21, 0.1, 0.7, 0.8])
    im = ax.imshow(J[np.ix_(order, order)], cmap="viridis", vmin=0, vmax=1, aspect="auto")
    ax.set_yticks(range(len(order)), [ids[i] for i in order], fontsize=6)
    ax.set_xticks(range(len(order)), [ids[i] for i in order], fontsize=6, rotation=90)
    ax.set_title("Interface residue-set similarity (Jaccard), hierarchical clustering")
    fig.colorbar(im, ax=ax, shrink=0.5, label="Jaccard")
    plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
m = s.dropna(subset=["Dimer interface area (Å²)"])
for meth, g in m.groupby("Method"):
    axes[0].scatter(g["Resolution (Å)"], g["Dimer interface area (Å²)"], label=meth, alpha=0.7)
axes[0].set(xlabel="Resolution (Å)", ylabel="Interface area (Å²)", title="Dimer interface area"); axes[0].legend(fontsize=8)
ifs = T["Interfaces"]; main = ifs[ifs.get("Main interface", False) == True]
freq = pd.Series([int(x) for v in main["Interface residues 1 (UniProt)"].dropna() for x in str(v).split(", ") if x]).value_counts().sort_index()
axes[1].bar(freq.index, freq.values / max(len(main), 1), width=1.0)
axes[1].set(xlabel="UniProt residue", ylabel="Fraction of entries", title="How often each residue is in the interface")
axes[1].axhline(cfg.get("interfaces.canonical.min_fraction"), ls="--", color="k", lw=0.8)
plt.tight_layout(); plt.show()
display(T["CrystalContacts"][T["CrystalContacts"].get("Matches canonical dimer", False) == True][
    ["PDB ID", "Lattice operator", "Canonical interface similarity", "BSA total (Å²)"]] if len(T["CrystalContacts"]) else None)

## Local experimental support

> **Decision — method-specific rules.**
> - **X-ray:** RSCC ≥ 0.90 and RSRZ ≤ 2 is *strong*; RSCC ≥ 0.80 and RSRZ ≤ 3 is *acceptable*.
> - **Cryo-EM:** a Q-score at or above the value expected for the map resolution is *strong*; up to 0.15 below it is *acceptable*. The expected value is Q = −0.1775·d + 1.1192 (Pintilie *et al.* 2020).
>
> RSCC and Q-score are never compared directly; only the categories are combined.
>
> The ligand/environment B-factor ratio from notebook 01 is kept as an *indirect* indicator only.

In [ ]:
lf = T["LocalModelFit"]
if "Group support score" in lf:
    hm = lf.pivot_table(index="PDB ID", columns="Group", values="Group support score", aggfunc="first")
    order = [p for p in T["TemplateRankings"].query("RankingMode == 'global'")["PDB ID"] if p in hm.index][:45]
    hm = hm.loc[order]
    fig, ax = plt.subplots(figsize=(1.2 + 0.9 * hm.shape[1], 0.22 * len(hm) + 1.5))
    cmap = plt.get_cmap("RdYlGn", 4).copy(); cmap.set_bad("lightgrey")
    im = ax.imshow(np.ma.masked_invalid(hm.values.astype(float)), cmap=cmap, vmin=-0.5, vmax=3.5, aspect="auto")
    ax.set_xticks(range(hm.shape[1]), hm.columns, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(hm)), hm.index, fontsize=7)
    cb = fig.colorbar(im, ax=ax, ticks=[0, 1, 2, 3], shrink=0.6)
    cb.ax.set_yticklabels(["unavailable", "weak", "acceptable", "strong"])
    ax.set_title("Local experimental support per residue group (top 45, global ranking order)")
    plt.tight_layout(); plt.show()
display(T["LigandModelFit"].query("Class == 'ligand of interest'").sort_values("Local support score").head(15))
display(T["EMMapFit"])

## Template rankings per modelling objective

Each mode is defined in `config/default_config.yaml` under `ranking.modes`, with its filters, criteria, directions, weights and missing-value policy.

Every ranked row keeps:
- the raw values of each criterion;
- each criterion's scaled component (0 = best, 1 = worst);
- a plain-text *Score explanation*, so every position can be audited.

> **Decision — no universal "best structure".** The best template for an MD start differs from the best open-lid or ligand-bound template. For example, `md_start` excludes:
> - entries with substitutions (such as the E424M of 9S7U);
> - entries with antibody or nanobody chains.

In [ ]:
top = T["RankingTop"]
display(top.pivot_table(index="Rank", columns="RankingMode", values="PDB ID", aggfunc="first"))
display(top.groupby("RankingMode")["Entries eligible"].first().to_frame())
r = T["TemplateRankings"]
display(r[r["RankingMode"] == "md_start"][["PDB ID", "Rank", "Score (0-1, higher = better)", "Score explanation"]].head(10))

## Milestone 2 checklist

| Component | Status |
|---|---|
| Assembly built from gemmi operators; identifiers and UniProt numbering preserved | ✔ |
| Interface contacts, typed interactions, BSA (FreeSASA or Shrake–Rupley, method recorded) | ✔ |
| Interface comparison: Jaccard, contact-pair similarity, recovery, interface RMSD, orientation, clustering, noncanonical flag | ✔ |
| Crystal-lattice contacts and canonical dimer detection | ✔ |
| wwPDB validation XML (any schema), LocalModelFit, LigandModelFit, EMMapFit | ✔ |
| Ten ranking modes with filters, weights and explanations | ✔ |
| Interface conservation | Milestone 3 (variants) |
| Conserved waters | Milestone 3 |